In [1]:
import tkinter as tk
from tkinter import ttk
import ipaddress
import re

# ===================== MODERN THEME CONFIGURATION =====================
# Define colors and fonts for a clean UI (Windows 11 / macOS inspired)

BG = "#F3F4F6"          # Main background color
CARD = "#FFFFFF"        # Card / input background
BTN = "#FFFFFF"         # Default button color
BTN_OP = "#E9ECEF"      # Operator / secondary button color
ACCENT = "#4F46E5"      # Primary accent color (buttons like "=")
TEXT = "#111827"        # Main text color
BORDER = "#E5E7EB"      # Border color

FONT = ("Segoe UI", 11)
FONT_BIG = ("Segoe UI", 24)
FONT_SMALL = ("Segoe UI", 10)

# ===================== CORE LOGIC =====================

def switch_mode():
    modes = ["Decimal", "Binary", "Hex"]
    current = mode_var.get()
    idx = (modes.index(current) + 1) % len(modes)
    mode_var.set(modes[idx])

def clear_input():
    input_entry.delete(0, tk.END)
    output_entry.delete(0, tk.END)

def safe_eval(value, base):
    value = value.upper()
    tokens = re.findall(r'[0-9A-F]+|[\+\-\*/]', value)
    expr = []
    for t in tokens:
        if t in ['+','-','*','/']:
            expr.append(t)
        else:
            # Convert from base → decimal
            expr.append(str(int(t, base)))
            
    # Evaluate final decimal expression
    return eval(" ".join(expr))

def calculate_only():
    value = input_entry.get().strip()
    mode = mode_var.get()

    try:
        if mode == "Hex":
            result = safe_eval(value, 16)
            res = hex(int(result))[2:].upper()

        elif mode == "Binary":
            result = safe_eval(value, 2)
            res = bin(int(result))[2:]

        else:
            res = str(eval(value))

        output_entry.delete(0, tk.END)
        output_entry.insert(0, res)

    except:
        output_entry.delete(0, tk.END)
        output_entry.insert(0, "Error")

def calculate_conversion(conversion):
    value = input_entry.get().strip()

    try:        
        # Detect input type automatically
        if any(c in value for c in "01") and re.fullmatch(r'[01+\-*/. ]+', value):
            result = safe_eval(value, 2)
        elif re.search(r'[A-F]', value.upper()):
            result = safe_eval(value, 16)
        else:
            result = eval(value)

        # Perform conversion
        if conversion == "Dec→Bin":
            res = bin(int(result))[2:]
        elif conversion == "Bin→Dec":
            res = str(int(result))
        elif conversion == "Dec→Hex":
            res = hex(int(result))[2:].upper()
        elif conversion == "Hex→Dec":
            res = str(int(result))
        elif conversion == "Bin→Hex":
            res = hex(int(result))[2:].upper()
        elif conversion == "Hex→Bin":
            res = bin(int(result))[2:]

        output_entry.delete(0, tk.END)
        output_entry.insert(0, res)

    except:
        output_entry.delete(0, tk.END)
        output_entry.insert(0, "Error")

# ===================== IP TOOL FUNCTIONS =====================

def clear_ip():
    ip_input.delete(0, tk.END)
    ip_output.delete(0, tk.END)
    for v in [ip_class_var, subnet_var, network_var, broadcast_var]:
        v.set("")

def validate_ip():
    ip = ip_input.get().strip()
    try:
        ipaddress.IPv4Address(ip)
        ip_output.delete(0, tk.END)
        ip_output.insert(0, "IP Valide")
        ip_output.config(fg="green")

        first_octet = int(ip.split(".")[0])

        # Determine class based on first octet
        if 1 <= first_octet <= 126:
            ip_class_var.set("Classe A")
            subnet_var.set("255.0.0.0")
        elif 128 <= first_octet <= 191:
            ip_class_var.set("Classe B")
            subnet_var.set("255.255.0.0")
        elif 192 <= first_octet <= 223:
            ip_class_var.set("Classe C")
            subnet_var.set("255.255.255.0")
        else:
            ip_class_var.set("Spéciale")
            subnet_var.set("255.255.255.0")
            
        # Calculate network & broadcast
        net = ipaddress.IPv4Network(f"{ip}/{subnet_var.get()}", strict=False)
        network_var.set(str(net.network_address))
        broadcast_var.set(str(net.broadcast_address))

    except:
        ip_output.delete(0, tk.END)
        ip_output.insert(0, "IP Invalide")
        ip_output.config(fg="red")

def ip_to_binary():
    ip = ip_input.get().strip()
    try:
        binary_str = ".".join([format(int(o), '08b') for o in ip.split('.')])
        ip_output.delete(0, tk.END)
        ip_output.insert(0, binary_str)
    except:
        ip_output.delete(0, tk.END)
        ip_output.insert(0, "IP Invalide")
        ip_output.config(fg="red")

# ===================== UI SETUP =====================

# Create main window
root = tk.Tk()
root.title("CS Calculator Pro")
root.geometry("420x680")
root.configure(bg=BG)
root.resizable(False, False)

# Configure ttk styles (tabs)
style = ttk.Style()
style.theme_use("clam")

style.configure("TNotebook", background=BG, borderwidth=0)
style.configure("TNotebook.Tab",
                background=BTN_OP,
                padding=[20, 10],
                font=("Segoe UI", 10))
style.map("TNotebook.Tab", background=[("selected", CARD)])

notebook = ttk.Notebook(root)
notebook.pack(expand=True, fill="both")

# ===================== BUTTON FACTORY =====================

def create_btn(parent, text, cmd, bg=BTN, fg=TEXT):
    return tk.Button(
        parent,
        text=text,
        command=cmd,
        bg=bg,
        fg=fg,
        font=FONT,
        bd=0,
        relief="flat",
        activebackground="#DDE1E7",
        cursor="hand2",
        highlightthickness=0,
        padx=10,
        pady=10
    )

# ===================== TAB 1: CALCULATOR =====================

tab1 = tk.Frame(notebook, bg=BG)
notebook.add(tab1, text="Calculator")

# Header with mode switch
header1 = tk.Frame(tab1, bg=BG)
header1.pack(fill="x", padx=12, pady=8)

mode_var = tk.StringVar(value="Decimal")

switch_btn = tk.Button(header1, text="⟳", command=switch_mode,
                        bg=BTN_OP, fg=TEXT, relief="flat",
                        font=("Segoe UI", 12), cursor="hand2")
switch_btn.pack(side="right")

# Display current mode
tk.Label(header1, textvariable=mode_var,
         bg=BG, fg=TEXT,
         font=("Segoe UI Semibold", 11)).pack(side="right", padx=8)

# Input & output fields
input_entry = tk.Entry(tab1, font=FONT_BIG, bd=0,
                       bg=CARD, fg=TEXT, justify="right")
input_entry.pack(fill="x", padx=15, pady=10)

output_entry = tk.Entry(tab1, font=("Segoe UI", 14),
                        bd=0, bg=CARD, fg=TEXT, justify="right")
output_entry.pack(fill="x", padx=15, pady=5)

# Button grid
grid1 = tk.Frame(tab1, bg=BG)
grid1.pack(expand=True, fill="both", padx=8, pady=10)

# Button layout
btns1 = [
['Dec→Bin','Bin→Dec','Dec→Hex','Hex→Dec'],
['7','8','9','C'],
['4','5','6','/'],
['1','2','3','*'],
['0','.','=','+']
]

# Generate buttons dynamically
for r in btns1:
    row = tk.Frame(grid1, bg=BG)
    row.pack(expand=True, fill="both")

    for b in r:
        bg, fg = BTN, TEXT
        cmd = lambda x=b: input_entry.insert(tk.END, x)

        if b == "C":
            cmd, bg = clear_input, BTN_OP
        elif b == "=":
            cmd, bg, fg = calculate_only, ACCENT, "white"
        elif "→" in b:
            cmd, bg = lambda x=b: calculate_conversion(x), BTN_OP
        elif b in ["/","*","+","-"]:
            bg = BTN_OP

        create_btn(row, b, cmd, bg, fg).pack(
            side="left", expand=True, fill="both", padx=4, pady=4
        )

# ===================== TAB 2: IP TOOLS =====================

tab2 = tk.Frame(notebook, bg=BG)
notebook.add(tab2, text="IP Tools")

# IP input/output fields
ip_input = tk.Entry(tab2, font=("Segoe UI", 16),
                    justify="center", bd=0, bg=CARD)
ip_input.pack(fill="x", padx=15, pady=10)

ip_output = tk.Entry(tab2, font=("Segoe UI", 12),
                     justify="center", bd=0, bg=CARD)
ip_output.pack(fill="x", padx=15, pady=5)

# Info display box
info_box = tk.Frame(tab2, bg=BG)
info_box.pack(fill="x", padx=15, pady=5)

# Variables to store computed info
ip_class_var = tk.StringVar()
subnet_var = tk.StringVar()
network_var = tk.StringVar()
broadcast_var = tk.StringVar()

def add_info(label, var):
    f = tk.Frame(info_box, bg=BG)
    f.pack(fill="x", pady=2)
    tk.Label(f, text=label, bg=BG, fg=TEXT, font=FONT_SMALL).pack(side="left")
    tk.Entry(f, textvariable=var, state="readonly",
             bg=BG, bd=0, fg=TEXT, font=FONT_SMALL,
             readonlybackground=BG).pack(side="right")

# Add info rows
add_info("CLASS", ip_class_var)
add_info("MASK", subnet_var)
add_info("NETWORK", network_var)
add_info("BROADCAST", broadcast_var)

# Buttons grid for IP tools
grid2 = tk.Frame(tab2, bg=BG)
grid2.pack(expand=True, fill="both", padx=8, pady=10)

btns2 = [
['VALIDATE','TO BINARY'],
['7','8','9','C'],
['4','5','6','Clear'],
['1','2','3','='],
['0','.']
]

# Generate buttons
for r in btns2:
    row = tk.Frame(grid2, bg=BG)
    row.pack(expand=True, fill="both")

    for b in r:
        bg, fg = BTN, TEXT
        cmd = lambda x=b: ip_input.insert(tk.END, x)

        if b == "VALIDATE":
            cmd, bg, fg = validate_ip, ACCENT, "white"
        elif b == "TO BINARY":
            cmd, bg = ip_to_binary, BTN_OP
        elif b == "Clear":
            cmd, bg = clear_ip, BTN_OP
        elif b == "C":
            cmd = lambda: ip_input.delete(len(ip_input.get())-1, tk.END)
            bg = BTN_OP
        elif b == "=":
            cmd, bg, fg = validate_ip, ACCENT, "white"

        create_btn(row, b, cmd, bg, fg).pack(
            side="left", expand=True, fill="both", padx=4, pady=4
        )

# ===================== START APPLICATION =====================

root.mainloop()